In [31]:
import pandas as pd
import glob 

annotators = [
    "../dataset_veronika",
    "../ukrainian_dataset",
]

annotator_dfs = {}
for i, annotator in enumerate(annotators):
    csv_files = glob.glob(f"{annotator}/*/*.csv")
    print(csv_files)
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        current_annotator_df = annotator_dfs.get(i, pd.DataFrame())
        annotator_dfs[i] = pd.concat([current_annotator_df, df], ignore_index=True)




['../dataset_veronika/test/_classes.csv', '../dataset_veronika/train/_classes.csv', '../dataset_veronika/valid/_classes.csv']
['../ukrainian_dataset/test/_classes.csv', '../ukrainian_dataset/train/_classes.csv', '../ukrainian_dataset/valid/_classes.csv']


In [32]:
import os 
for image_file in glob.glob("./ukrainian/images/*"):
    file_name = image_file.split("/")[-1].split("_jpg")[0]
    os.rename(image_file, f"./ukrainian/images/{file_name}")

In [33]:
import difflib


def get_label_mapping(current_labels, expected_labels, cutoff=0.8):
    label_mapping = {}
    for label in current_labels:
        if label == "No Propaganda":
            label_mapping[label] = "No Propaganda"
            continue
        matches = get_close_matches(label, expected_labels, n=1, cutoff=cutoff)
        if matches:
            label_mapping[label] = matches[0]
            expected_labels.remove(matches[0])
        else:
            label_mapping[label] = None

    return label_mapping

def get_close_matches(word : str, possibilities : list[str], n=1, cutoff=0.6):
    return difflib.get_close_matches(word, possibilities, n=n, cutoff=cutoff)


In [34]:
expected_labels = {'Appeal to (Strong) Emotions',
 'Appeal to authority',
 'Appeal to fear/prejudice',
 'Bandwagon',
 'Black-and-white Fallacy/Dictatorship',
 'Causal Oversimplification',
 'Doubt',
 'Exaggeration/Minimisation',
 'Flag-waving',
 'Glittering generalities (Virtue)',
 'Loaded Language',
 "Misrepresentation of Someone's Position (Straw Man)",
 'Name calling/Labeling',
 'Obfuscation, Intentional vagueness, Confusion',
 'Presenting Irrelevant Data (Red Herring)',
 'Reductio ad hitlerum',
 'Repetition',
 'Slogans',
 'Smears',
 'Thought-terminating cliché',
 'Transfer',
 'Whataboutism'}


old_to_new_label_mapping = get_label_mapping(list(annotator_dfs[0].iloc[:,1:].columns), expected_labels, cutoff=0.1)



In [35]:
old_to_new_label_mapping

{'Appeal to -Strong- Emotions': 'Appeal to (Strong) Emotions',
 'Appeal to fear-prejudice': 'Appeal to fear/prejudice',
 'Bandwagon': 'Bandwagon',
 'Black-and-white Fallacy-Dictatorship': 'Black-and-white Fallacy/Dictatorship',
 'Causal Oversimplification': 'Causal Oversimplification',
 'Doubt': 'Doubt',
 'Exaggeration-Minimisation': 'Exaggeration/Minimisation',
 'Flag-waving': 'Flag-waving',
 'Glittering generalities -Virtue-': 'Glittering generalities (Virtue)',
 'Loaded Language': 'Loaded Language',
 'Misrepresentation of Someones Position -Straw Man-': "Misrepresentation of Someone's Position (Straw Man)",
 'Name calling-Labeling': 'Name calling/Labeling',
 'No Propaganda': 'No Propaganda',
 'Obfuscation- Intentional vagueness- Confusion': 'Obfuscation, Intentional vagueness, Confusion',
 'Presenting Irrelevant Data -Red Herring-': 'Presenting Irrelevant Data (Red Herring)',
 'Reductio ad hitlerum': 'Reductio ad hitlerum',
 'Repetition': 'Repetition',
 'Slogans': 'Slogans',
 'Smear

In [36]:
for annotator_df in annotator_dfs.values():
    annotator_df.rename(columns=old_to_new_label_mapping, inplace=True)

In [37]:
# prepare jsonl files for each annotator
import json

annotators_jsonl = []
for annotator_id, annotator_df in annotator_dfs.items():
    output_data = []
    columns = annotator_df.columns[1:]
    for _, row in annotator_df.iterrows():
        image_name = row[0].split("_jpg")[0]
        labels = columns[row[1:] == 1].tolist()
        output_data.append({
            "image": image_name,
            "labels": labels
        })

    annotators_jsonl.append(output_data)


/tmp/ipykernel_72010/1449656344.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  image_name = row[0].split("_jpg")[0]


In [38]:
df1 = pd.DataFrame.from_records(annotators_jsonl[0])
df2 = pd.DataFrame.from_records(annotators_jsonl[1])


annotators_df = df1.merge(df2, on="image", how="outer", suffixes=('_annotator1', '_annotator2'))
annotators_df.sort_values(by="image", inplace=True)
annotators_df[-30:]

,image,labels_annotator1,labels_annotator2
170,tg_5344,[No Propaganda],"[Causal Oversimplification, Transfer]"
171,tg_5345,[No Propaganda],"[Appeal to (Strong) Emotions, Loaded Language]"
172,tg_5346,"[Appeal to fear/prejudice, Causal Oversimplifi...","[Appeal to fear/prejudice, Causal Oversimplifi..."
173,tg_5347,"[Exaggeration/Minimisation, Flag-waving]","[Exaggeration/Minimisation, Flag-waving, Loade..."
174,tg_5349,[No Propaganda],[No Propaganda]
175,tg_5350,"[Appeal to (Strong) Emotions, Black-and-white ...","[Appeal to (Strong) Emotions, Causal Oversimpl..."
176,tg_5351,[No Propaganda],[No Propaganda]
177,tg_5352,[No Propaganda],[No Propaganda]
178,tg_5353,[No Propaganda],[No Propaganda]
179,tg_5354,[No Propaganda],[No Propaganda]


In [39]:
annotators_df.to_csv("/home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/annotators.csv", index=False)

In [40]:
#out format
#{"id": "858_batch_2", "labels": ["Flag-waving", "Transfer"], "text": "ВИБОРИ 2020: ОБЕРИ ФАКЕЛ", "image": "858_image_batch_2.png"}
#
with open('./ukrainian/raw/images_data.json','r') as f:
    data = json.load(f)

data
get_file_name = lambda x: x.split("/")[-1]
data = [{'text': item['text'], 'image': get_file_name(item['path'])} for item in data]

In [41]:
annotations = sorted(annotators_jsonl[0], key=lambda x: x['image'])
data  = sorted(data, key=lambda x: x['image'])


In [42]:
results = []
for annotation, item in zip(annotations, data):
    results.append({
        "id": item['image'].split(".")[0],
        "labels": annotation['labels'],
        "text": item['text'],
        "image": item['image']
    })

In [43]:
results

[{'id': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig',
  'labels': ['Smears'],
  'text': 'Бойові виходи за лінію фронту',
  'image': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig.jpg'},
 {'id': 'csv_10nuP18xGKwhwzie8LQg1jJvQ7sEyBUNk',
  'labels': ['Whataboutism'],
  'text': 'Невідомий художник. Дівчина з тараканами. -- Геніальна ідея! Українці повністю підтримають! Симоньян пропонує "здійснити ядерний вибух десь над Сибіром"',
  'image': 'csv_10nuP18xGKwhwzie8LQg1jJvQ7sEyBUNk.jpg'},
 {'id': 'csv_11FXDzESQrZpC5yqpUrKbsipXldf1owRH',
  'labels': ['Causal Oversimplification', 'Smears'],
  'text': '- Німеччина передає ЗРК Patriot. - На наступний день хтось збиває метеорит. - Хлопці *не тицяли ту кнопку*.',
  'image': 'csv_11FXDzESQrZpC5yqpUrKbsipXldf1owRH.jpg'},
 {'id': 'csv_12eKY_11DKhS3OJkRBe3Ik-w6IF0XoKHP',
  'labels': ['Doubt', 'Smears'],
  'text': 'Сьогодні побачимо, як працює 5 стаття Статуту NATO?',
  'image': 'csv_12eKY_11DKhS3OJkRBe3Ik-w6IF0XoKHP.jpg'},
 {'id': 'csv_12vuewEZ3Dv50-FjDlCv4RFJpRr-

In [46]:
import base64
import requests
import json
from openai import OpenAI
import time 

IMAGE_PATH = "../datasets/propaganda_950/images/1_image.png"

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
    timeout=3600
)

def inference_model(prompt, image_path):
    # Load image & base64 encode
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image_url", 
                 "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                    },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    response = client.chat.completions.create(
        model="Qwen/Qwen3-VL-8B-Instruct-FP8",
        messages=messages,
        max_tokens=2048
    )
    return response.choices[0].message.content

prompt = "Describe the image in detail."
start = time.time()
response = inference_model(prompt, IMAGE_PATH)

print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {response}")


Response costs: 16.39s
Generated text: This image is a political infographic designed as a side-by-side comparison, contrasting economic performance during the presidencies of Barack Obama and Donald Trump. It uses contrasting colors, text, and imagery to present its claims.

The image is divided vertically into two equal halves.

**Left Side: "8 YEARS UNDER OBAMA"**
*   **Header:** The top section has a dark gray background with the white text "8 YEARS UNDER OBAMA".
*   **Image:** Below the header is a photograph of President Barack Obama, looking serious and slightly to his right. He is wearing a dark suit, white shirt, and a red patterned tie.
*   **Data Boxes (Red Background, White Text):** Three red rectangular boxes are stacked vertically below the image, presenting negative economic figures:
    *   "4 MILLION JOBS LOST"
    *   "UNEMPLOYMENT PEAKED TO 9.9%"
    *   "GDP DOWN 2.8%"

**Right Side: "2 ½ YEARS UNDER TRUMP"**
*   **Header:** The top section has a black background wi

In [47]:
prompt = """
You are OCR assistant. Your task is to extract text from the image and provide it in the output.
Please provide only text that is on the image. If no text is present, respond with 'No_text'.

Text on the image: 
"""

In [48]:
from tqdm import tqdm
for item in tqdm(results):
    image_path = f"./ukrainian/images/{item['image']}"
    if item['text'].strip().lower() == "unknown":
        ocr_text = inference_model(prompt, image_path)
        if ocr_text.strip().lower() != "no_text":
            item['text'] = ocr_text


100%|██████████| 200/200 [02:06<00:00,  1.58it/s]


In [49]:
results[0]

{'id': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig',
 'labels': ['Smears'],
 'text': 'Бойові виходи за лінію фронту',
 'image': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig.jpg'}

In [53]:
# ── Parse final labels from adjudicator CSV ──────────────────────────────────
final_csv_path = "/home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/annotators - annotators.csv"
orig_csv_path = "/home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/annotators.csv"

orig_df = pd.read_csv(orig_csv_path)
final_df = pd.read_csv(final_csv_path)

# image_N.jpg → real image name (same row order)
image_name_to_real = dict(zip(final_df["image"], orig_df["image"]))

import csv as csv_mod

def parse_final_labels(final_str):
    s = str(final_str).strip()
    if not s or s == "No Propaganda" or s == "nan":
        return []
    # skipinitialspace=True so ', "Obfuscation...' is treated as a quoted field
    return [l.strip().strip("'") for l in next(csv_mod.reader([s], skipinitialspace=True)) if l.strip()]

# Collect all unique raw labels
raw_labels = set()
for s in final_df["final"]:
    raw_labels.update(parse_final_labels(s))

print("Raw labels from final column:")
for l in sorted(raw_labels):
    print(f"  {l}")

Raw labels from final column:
  Appeal to (Strong) emotions
  Appeal to authority
  Appeal to fear/prejudice
  Bandwagon
  Black-and-white Fallacy/Dictatorship
  Causal Oversimplification
  Doubt
  Exaggeration/Minimisation
  Flag-waiving
  Glittering generalities (Virtue)
  Loaded Language
  Misrepresentation of Someone's Position (Straw Man)
  Name calling/Labeling
  Obfuscation, Intentional vagueness, Confusion
  Presenting Irrelevant Data (Red Herring)
  Reductio ad hitlerum
  Repetitions
  Slogans
  Smears
  Thought-terminating cliché
  Transfer


In [54]:
final_df

,image,labels_annotator1,labels_annotator2,final
0,image_0.jpg,['Smears'],"['Appeal to (Strong) Emotions', 'Flag-waving',...","Flag-waiving, Smears, Transfer"
1,image_1.jpg,['Whataboutism'],"['Appeal to (Strong) Emotions', 'Appeal to fea...","Smears, Transfer"
2,image_2.jpg,"['Causal Oversimplification', 'Smears']","['Appeal to (Strong) Emotions', 'Exaggeration/...","Exaggeration/Minimisation, Flag-waiving"
3,image_3.jpg,"['Doubt', 'Smears']","['Appeal to (Strong) Emotions', 'Doubt', 'Smea...","Doubt, Smears"
4,image_4.jpg,"['Appeal to fear/prejudice', 'Exaggeration/Min...","['Appeal to fear/prejudice', 'Exaggeration/Min...","Appeal to fear/prejudice, Smears, Exaggeration..."
...,...,...,...,...
195,image_195.jpg,['No Propaganda'],['No Propaganda'],No Propaganda
196,image_196.jpg,['No Propaganda'],['Appeal to (Strong) Emotions'],Appeal to (Strong) emotions
197,image_197.jpg,['No Propaganda'],['No Propaganda'],No Propaganda
198,image_198.jpg,['No Propaganda'],['No Propaganda'],No Propaganda


In [55]:
# ── Map dirty labels → canonical using difflib ────────────────────────────────
# Re-define expected_labels (the original set was mutated by earlier get_label_mapping calls)
expected_labels_fresh = [
    'Appeal to (Strong) Emotions', 'Appeal to authority', 'Appeal to fear/prejudice',
    'Bandwagon', 'Black-and-white Fallacy/Dictatorship', 'Causal Oversimplification',
    'Doubt', 'Exaggeration/Minimisation', 'Flag-waving',
    'Glittering generalities (Virtue)', 'Loaded Language',
    "Misrepresentation of Someone's Position (Straw Man)", 'Name calling/Labeling',
    'Obfuscation, Intentional vagueness, Confusion',
    'Presenting Irrelevant Data (Red Herring)', 'Reductio ad hitlerum',
    'Repetition', 'Slogans', 'Smears', 'Thought-terminating cliché',
    'Transfer', 'Whataboutism',
]

final_label_mapping = get_label_mapping(sorted(raw_labels), list(expected_labels_fresh), cutoff=0.6)

print("Final label mapping:")
for old, new in sorted(final_label_mapping.items()):
    flag = " UNMAPPED" if new is None else ("" if old == new else " <- fixed")
    print(f"  {old!r:60s} -> {new!r}{flag}")


Final label mapping:
  'Appeal to (Strong) emotions'                                -> 'Appeal to (Strong) Emotions' <- fixed
  'Appeal to authority'                                        -> 'Appeal to authority'
  'Appeal to fear/prejudice'                                   -> 'Appeal to fear/prejudice'
  'Bandwagon'                                                  -> 'Bandwagon'
  'Black-and-white Fallacy/Dictatorship'                       -> 'Black-and-white Fallacy/Dictatorship'
  'Causal Oversimplification'                                  -> 'Causal Oversimplification'
  'Doubt'                                                      -> 'Doubt'
  'Exaggeration/Minimisation'                                  -> 'Exaggeration/Minimisation'
  'Flag-waiving'                                               -> 'Flag-waving' <- fixed
  'Glittering generalities (Virtue)'                           -> 'Glittering generalities (Virtue)'
  'Loaded Language'                                       

In [56]:
# ── Build results with final labels + OCR text, save JSONL ────────────────────
id_to_result = {r["image"]: r for r in results}

final_results = []
for _, row in final_df.iterrows():
    real_image = image_name_to_real.get(row["image"], row["image"])

    raw = parse_final_labels(row["final"])
    mapped = [final_label_mapping.get(l, l) for l in raw]
    mapped = [l for l in mapped if l is not None]

    existing = id_to_result.get(real_image, {})
    text = existing.get("text", "")

    final_results.append({
        "id": real_image.split(".")[0],
        "labels": mapped,
        "text": text,
        "image": real_image,
    })

print(f"Built {len(final_results)} records")
print(f"Example: {final_results[0]}")

out_path = "/home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/test.jsonl"
with open(out_path, "w") as f:
    for item in final_results:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")
print(f"Saved to {out_path}")


Built 200 records
Example: {'id': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig', 'labels': ['Flag-waving', 'Smears', 'Transfer'], 'text': '', 'image': 'csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig'}
Saved to /home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/test.jsonl


In [58]:
# ── Add OCR text from results to existing JSONL ──────────────────────────────
jsonl_path = "/home/nazara/Data2/DIPLOMA/datasets/ukrainian/annotations/test.jsonl"

with open(jsonl_path) as f:
    records = [json.loads(line) for line in f if line.strip()]

# Build lookup from results (which already has OCR text)
id_to_text = {r["id"]: r["text"] for r in results}

for item in records:
    if not item["image"].endswith(".jpg"):
        item["image"] = item["image"] + ".jpg"
    if not item["text"].strip():
        item["text"] = id_to_text.get(item["id"], "")

with open(jsonl_path, "w") as f:
    for item in records:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

n_with_text = sum(1 for r in records if r["text"].strip())
print(f"Updated {len(records)} records ({n_with_text} with text)")

Updated 200 records (200 with text)


In [ ]:
import os 
import json 


path = "/home/nazara/Data2/DIPLOMA/datasets/translated/annotations/test_whole.jsonl"
test_data = []
with open(path, "r") as f:
    for line in f:
        obj = json.loads(line)
        if os.path.exists('/home/nazara/Data2/DIPLOMA/datasets/translated/images/' + obj['image']):
            test_data.append(obj)


In [ ]:
#save jsonl
with open('/home/nazara/Data2/DIPLOMA/datasets/translated/annotations/test.jsonl', 'w') as f:
    for item in test_data:
        json.dump(item, f)
        f.write('\n')


In [ ]:
annotators_df

,image,labels_annotator1,labels_annotator2
0,csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig.jpg,[Smears],"[Appeal to (Strong) Emotions, Flag-waving, Sme..."
1,csv_10nuP18xGKwhwzie8LQg1jJvQ7sEyBUNk.jpg,[Whataboutism],"[Appeal to (Strong) Emotions, Appeal to fear/p..."
2,csv_11FXDzESQrZpC5yqpUrKbsipXldf1owRH.jpg,"[Causal Oversimplification, Smears]","[Appeal to (Strong) Emotions, Exaggeration/Min..."
3,csv_12eKY_11DKhS3OJkRBe3Ik-w6IF0XoKHP.jpg,"[Doubt, Smears]","[Appeal to (Strong) Emotions, Doubt, Smears]"
4,csv_12vuewEZ3Dv50-FjDlCv4RFJpRr-EhPNZ.jpg,"[Appeal to fear/prejudice, Exaggeration/Minimi...","[Appeal to fear/prejudice, Exaggeration/Minimi..."
...,...,...,...
195,tg_5374.jpg,[No Propaganda],[No Propaganda]
196,tg_5375.jpg,[No Propaganda],[Appeal to (Strong) Emotions]
197,tg_5376.jpg,[No Propaganda],[No Propaganda]
198,tg_5379.jpg,[No Propaganda],[No Propaganda]


In [ ]:
import shutil


AttributeError: module 'os' has no attribute 'copy'

In [ ]:
annotators_df

,image,labels_annotator1,labels_annotator2,image_name
0,csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig.jpg,[Smears],"[Appeal to (Strong) Emotions, Flag-waving, Sme...",image_0.jpg
1,csv_10nuP18xGKwhwzie8LQg1jJvQ7sEyBUNk.jpg,[Whataboutism],"[Appeal to (Strong) Emotions, Appeal to fear/p...",image_1.jpg
2,csv_11FXDzESQrZpC5yqpUrKbsipXldf1owRH.jpg,"[Causal Oversimplification, Smears]","[Appeal to (Strong) Emotions, Exaggeration/Min...",image_2.jpg
3,csv_12eKY_11DKhS3OJkRBe3Ik-w6IF0XoKHP.jpg,"[Doubt, Smears]","[Appeal to (Strong) Emotions, Doubt, Smears]",image_3.jpg
4,csv_12vuewEZ3Dv50-FjDlCv4RFJpRr-EhPNZ.jpg,"[Appeal to fear/prejudice, Exaggeration/Minimi...","[Appeal to fear/prejudice, Exaggeration/Minimi...",image_4.jpg
...,...,...,...,...
195,tg_5374.jpg,[No Propaganda],[No Propaganda],image_195.jpg
196,tg_5375.jpg,[No Propaganda],[Appeal to (Strong) Emotions],image_196.jpg
197,tg_5376.jpg,[No Propaganda],[No Propaganda],image_197.jpg
198,tg_5379.jpg,[No Propaganda],[No Propaganda],image_198.jpg


In [ ]:
import shutil
annotators_df['image_name'] = [f"image_{x}.jpg" for x in list(range(len(annotators_df)))]
for _, row in annotators_df.iterrows():
    image_name = row['image_name']
    source_path = f"./ukrainian/images/{row['image']}"
    target_path = f"./ukrainian/selection/{image_name}"
    shutil.copy(source_path, target_path)
annotators_df['image'] = annotators_df['image_name']
annotators_df.drop(columns=['image_name'], inplace=True)
annotators_df.to_csv("/home/nazara/Data2/DIPLOMA/datasets/ukrainian/selection/annotators.csv", index=False)

{0:                                               filename  \
 0    tg_5293_jpg.rf.6079fcdff12ba72161ccee81de7cbe7...   
 1    csv_1cmjkZ8nBDuRrOEk4CzrF69nLdbIOw_yK_jpg.rf.9...   
 2    tg_5365_jpg.rf.e7365586f7b40105a3b697902ec7f74...   
 3    tg_5283_jpg.rf.0d35e8e61c4dcf661f9d666e19e487a...   
 4    csv_1KXdSpbH06J7508iSvl9xeRvbZ3vXhF3E_jpg.rf.e...   
 ..                                                 ...   
 195  csv_1UTsSKtviwlAP0yIT6U5hbn1oTUgZsV46_jpg.rf.0...   
 196  csv_1Py92xnYDI5ilr30qKDF1GIIx9ftRV5h-_jpg.rf.7...   
 197  csv_16iNP31Z76-aUG4_3v2dVQXT6-vl5WvrT_jpg.rf.0...   
 198  tg_5334_jpg.rf.0ab14c7ec2f0180549a7e103a8f87b0...   
 199  csv_1yNAKOfqD7K-rC-y5zsEtYyuB3xMvXbdX_jpg.rf.c...   
 
      Appeal to (Strong) Emotions  Appeal to fear/prejudice  Bandwagon  \
 0                              0                         0          0   
 1                              1                         0          0   
 2                              0                         1      

In [ ]:
source_path

'./ukrainian/images/csv_10RGn0cWAS8GFV0dlIwylPDcbmJNfLAig.jpg'